[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SysBioChalmers/MESBcourse/blob/main/exercises/GEM1_raven/gem1.ipynb)

# Day 1: Genome-scale models in cobrapy

A Python / cobrapy port of the RAVEN MATLAB exercise (`gem1.mlx`).

**Learning goals**

- Add new reactions to an existing GEM to facilitate synthesis of a product of interest.
- Calculate maximum biochemical yields.
- Interpret production envelope and phenotypic phase plane plots.
- Perform FSEOF and interpret the results.
- Perform optKnock and interpret the results.

## Setup

Required packages: `cobra`, `pandas`, `numpy`, `matplotlib`, `openpyxl`, `scipy`.

```bash
pip install cobra pandas numpy matplotlib openpyxl scipy
```

> **Note on RAVEN vs cobrapy.** The original exercise uses the RAVEN Toolbox
> (MATLAB). RAVEN ships dedicated functions such as `runProductionEnvelope`,
> `runDynamicFBA`, `FSEOF` and `runSimpleOptKnock`. cobrapy does not provide
> all of these out of the box, so below we define small helper functions that
> reproduce the RAVEN behaviour. The LP solutions themselves are identical
> (both call a standard LP solver), so the numerical results match the RAVEN
> answer key.

In [ ]:
import sys
!{sys.executable} -m pip install cobra pandas numpy matplotlib openpyxl scipy "git+https://github.com/SysBioChalmers/raven-python.git@main"
!wget -q https://raw.githubusercontent.com/SysBioChalmers/MESBcourse/refs/heads/main/exercises/GEM1_raven/iECD_1391.xml
!wget -q https://raw.githubusercontent.com/SysBioChalmers/MESBcourse/refs/heads/main/exercises/GEM1_raven/knockout_SelectedRxns.mat

In [ ]:
import cobra
from cobra import Metabolite, Reaction
from cobra.io import read_sbml_model, write_sbml_model
from cobra.flux_analysis import pfba
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio
from raven_python.analysis import fseof

print('cobra version:', cobra.__version__)

## 1. Add reactions and simulate production

For this exercise we study the production of **biliverdin** in *E. coli*.
Biliverdin is a derivative of heme with a range of potential protective
effects (anti-mutagenic, antioxidant). Heme oxygenase catalyses the
formation of biliverdin from heme, releasing iron and carbon monoxide.
*E. coli* is not a native producer of biliverdin, but overexpression of a
gene from the cyanobacterium *Synechocystis* spp. enables the reaction:

$$\text{heme} + 3\,\text{NADPH} + 3\,\text{O}_2 \rightarrow \text{biliverdin} + \text{Fe}^{2+} + \text{CO} + 3\,\text{NADP}^+ + 3\,\text{H}_2\text{O}$$

We will introduce this reaction and run some analyses around its production.
First we load the genome-scale model of *E. coli* (`iECD_1391`). In cobrapy
the SBML reader strips the `M_`/`R_` SBML prefixes, so metabolite and
reaction identifiers match the RAVEN ones (e.g. `pheme_c`, `EX_glc`).

In [ ]:
model = read_sbml_model('iECD_1391.xml')
print(model)
print(f'Reactions:   {len(model.reactions)}')
print(f'Metabolites: {len(model.metabolites)}')
print(f'Genes:       {len(model.genes)}')

### 1.1 Add the required reactions

When new reactions are added it is essential that no unnecessary duplicate
metabolites are introduced. After checking which metabolites are already
present (and noting their identifiers), it turns out that neither biliverdin
nor carbon monoxide is in the model yet. These need to be added — but in
which compartments? The model has three compartments:

In [ ]:
print(model.compartments)

We assume carbon monoxide can diffuse through the periplasmic membrane and
is excreted (so we add a transport chain and an exchange reaction).
Biliverdin mostly stays intracellular, but to let it accumulate we add an
exchange reaction that directly drains the cytoplasmic biliverdin.

First we add the metabolites, each with a unique identifier. For simplicity
we leave out additional data such as the chemical formula, charge, and
database annotations (KEGG, MetaCyc, ...). None of that is required to add
the metabolites:

In [ ]:
co_c = Metabolite('co_c', name='carbon monoxide', compartment='c')
co_p = Metabolite('co_p', name='carbon monoxide', compartment='p')
co_e = Metabolite('co_e', name='carbon monoxide', compartment='e')
biliverdin_c = Metabolite('biliverdin_c', name='biliverdin', compartment='c')

model.add_metabolites([co_c, co_p, co_e, biliverdin_c])
print('Metabolites now in model:', len(model.metabolites))

Next we add the heme oxygenase reaction (`HEMEOX`) plus the required
transport and exchange reactions. In cobrapy an irreversible reaction is
written with `-->` and a reversible one with `<=>`. A reaction with only a
left-hand side (e.g. `biliverdin_c -->`) is an exchange/sink reaction.
`build_reaction_from_string` looks up the metabolite IDs we just added.

In [ ]:
reactions_to_add = {
    'HEMEOX':        'pheme_c + 3 nadph_c + 5 h_c + 3 o2_c --> biliverdin_c + fe2_c + co_c + 3 nadp_c + 3 h2o_c',
    'EX_biliverdin': 'biliverdin_c -->',
    'COtpp':         'co_c <=> co_p',
    'COtex':         'co_p <=> co_e',
    'EX_co':         'co_e <=>',
}
for rxn_id, equation in reactions_to_add.items():
    rxn = Reaction(rxn_id)
    model.add_reactions([rxn])
    rxn.build_reaction_from_string(equation)

for rxn_id in reactions_to_add:
    rxn = model.reactions.get_by_id(rxn_id)
    print(f'{rxn_id:14s} {rxn.reaction}   bounds={rxn.bounds}')

Finally we set a few constraints and make sure biliverdin export is the
objective function. We then save the modified model so the later sections
can reload it (mirroring RAVEN's `exportModel`).

In [ ]:
model.reactions.EX_glc.lower_bound = -10   # max glucose uptake 10 mmol/gDCW/h
model.reactions.EX_o2.lower_bound  = -20   # max oxygen uptake 20 mmol/gDCW/h
model.objective = 'EX_biliverdin'

write_sbml_model(model, 'iECD_1391_biliverdin.xml')
print('Wrote iECD_1391_biliverdin.xml')

### 1.2 Calculate maximum biochemical yield

Given the maximum glucose uptake set above, what is the maximum biochemical
yield of biliverdin? With biliverdin export as the objective, run FBA:

In [ ]:
model.objective = 'EX_biliverdin'
solution = model.optimize()
print(f'Maximum biliverdin production is: {solution.objective_value:.4f}')

### Question 1

**Question 1: What is the maximum biochemical biliverdin yield in *E. coli*? (Think about the units!)**

### 1.3 Plot production envelope

cobrapy does not plot a RAVEN-style production envelope directly, so we
define a small helper that mirrors `runProductionEnvelope`: it scans a range
of fixed growth rates and, at each, finds the minimum and maximum target
flux.

In [ ]:
def production_envelope(model, target_rxn, biomass_rxn, n_pts=20, plot=True):
    """Byproduct secretion envelope (mirrors RAVEN runProductionEnvelope).
    Returns (biomass_values, target_lower, target_upper)."""
    m = model.copy()
    m.objective = biomass_rxn
    growth_max = m.slim_optimize()
    m.objective_direction = 'min'
    growth_min = m.slim_optimize()

    biomass_values = np.linspace(growth_min, growth_max, n_pts)
    lower = np.full(n_pts, np.nan)
    upper = np.full(n_pts, np.nan)

    m.objective = target_rxn
    br = m.reactions.get_by_id(biomass_rxn)
    for i, bv in enumerate(biomass_values):
        br.bounds = (bv, bv)               # fix growth rate
        m.objective_direction = 'max'
        upper[i] = m.slim_optimize()
        m.objective_direction = 'min'
        lower[i] = m.slim_optimize()

    if plot:
        plt.figure(figsize=(6, 4))
        plt.plot(np.concatenate([biomass_values, biomass_values[::-1]]),
                 np.concatenate([upper, lower[::-1]]), 'b', linewidth=2)
        plt.xlabel('Growth rate (1/h)')
        plt.ylabel(target_rxn.replace('_', '-') + ' (mmol/gDW h)')
        plt.title('Production envelope')
        plt.tight_layout()
        plt.show()
    return biomass_values, lower, upper

In [ ]:
biomass_values, target_lower, target_upper = production_envelope(
    model, 'EX_biliverdin', 'Ec_biomass_iJO1366_core_53p95M', n_pts=25)

### Question 2

**Question 2: Describe what you see in the graph.**

You have now seen how to take an existing model and modify it to simulate
the synthesis of a heterologous product. While the mechanics are
straightforward, a few critical points deserve attention. Imagine you work
on a project where you identified a gene cluster responsible for producing a
new plant secondary metabolite. You don't know exactly what each gene does,
but together they make this interesting compound. You plan to express the
whole gene cluster in yeast, assuming it constitutes the complete pathway,
and you want to estimate the maximum biochemical yield in yeast.

### Question 3

**Question 3: What challenges will you encounter when you add the required reactions to the model?**

## 2. Dynamic flux balance analysis (dFBA)

`HEMEOX` is a heterologous reaction expressed from a *Synechocystis* gene,
and its activity can be tuned (e.g. by changing the promoter). In the model
we mimic this by changing the bounds of `HEMEOX`, and we observe the effect
on the biliverdin titer reached during a batch cultivation, using **dynamic
FBA**. To avoid confusion with any changes made above, we reload the model
that was saved in Section 1.

In [ ]:
model = read_sbml_model('iECD_1391_biliverdin.xml')

### 2.1 Set parameters for dynamic FBA

With 200 mM glucose we simulate biliverdin production over 5 hours. cobrapy
has no built-in dynamic FBA, so we define a helper using the *static
optimization approach* — exactly the algorithm RAVEN's `runDynamicFBA` uses
(itself adapted from the COBRA Toolbox). At each time step it (1) caps each
uptake by the available concentration, (2) maximises growth with FBA, and
(3) updates biomass and metabolite concentrations analytically.

> RAVEN's `runDynamicFBA` minimises total flux (`solveLP(model, 1)`). Here
> plain FBA gives identical dynamics — the growth objective already fixes the
> glucose uptake and the (forced) `HEMEOX`/biliverdin flux — and is ~10×
> faster, so we use it.

In [ ]:
def run_dynamic_fba(model, substrate_rxns, init_concentrations, init_biomass,
                    time_step, n_steps, excl_uptake_rxns, biomass_rxn='EX_biomass'):
    """Dynamic FBA, static optimization approach (mirrors RAVEN runDynamicFBA).
    Returns (conc_matrix, exc_rxn_names, time_vec, biomass_vec)."""
    model = model.copy()
    if isinstance(substrate_rxns, str):
        substrate_rxns = [substrate_rxns]
    init_concentrations = np.atleast_1d(init_concentrations).astype(float)

    # exchange (boundary) reactions, minus the ones we treat as non-limiting
    exc = [r.id for r in model.boundary if r.id not in excl_uptake_rxns]
    conc = np.zeros(len(exc))
    for sr, ic in zip(substrate_rxns, init_concentrations):
        conc[exc.index(sr)] = ic

    original_bound = np.array([-model.reactions.get_by_id(r).lower_bound for r in exc])
    # open uptakes without a given concentration are assumed non-limiting
    conc[(conc == 0) & (original_bound > 0)] = 1000.0
    biomass = float(init_biomass)

    def set_uptake_bounds(conc, biomass):
        ub = conc / (biomass * time_step)
        ub[ub > 1000] = 1000
        above = (ub > original_bound) & (original_bound > 0)
        ub[above] = original_bound[above]
        ub[np.abs(ub) < 1e-9] = 0
        for r, b in zip(exc, ub):
            model.reactions.get_by_id(r).lower_bound = -b

    set_uptake_bounds(conc, biomass)
    conc_matrix, biomass_vec, time_vec = [conc.copy()], [biomass], [0.0]

    for step in range(n_steps):
        sol = model.optimize()             # maximise growth
        if sol.status != 'optimal':
            print(f'No feasible solution - nutrients exhausted. Biomass: {biomass:.4f}')
            break
        mu = sol.fluxes[biomass_rxn]
        if mu is None or mu <= 1e-9:
            print(f'No growth - nutrients exhausted. Biomass: {biomass:.4f}')
            break
        uptake_flux = np.array([sol.fluxes[r] for r in exc])
        biomass = biomass * np.exp(mu * time_step)
        conc = conc - uptake_flux / mu * biomass * (1 - np.exp(mu * time_step))
        conc[conc <= 0] = 0
        biomass_vec.append(biomass)
        conc_matrix.append(conc.copy())
        time_vec.append((step + 1) * time_step)
        set_uptake_bounds(conc, biomass)

    conc_matrix = np.array(conc_matrix).T            # rows = reactions, cols = time
    keep = np.any(conc_matrix > 0, axis=1)
    return conc_matrix[keep], [e for e, k in zip(exc, keep) if k], np.array(time_vec), np.array(biomass_vec)

In [ ]:
substrate_rxns       = 'EX_glc'   # substrate whose concentration changes
init_concentrations  = 200        # initial glucose (mmol/L)
init_biomass         = 0.01       # initial biomass (g/L)
time_step            = 0.25       # length of each step (h)
n_steps              = 20         # 20 steps x 0.25 h = 5 h
plot_rxn             = 'EX_biliverdin'
excl_uptake_rxns     = ['EX_co2', 'EX_o2', 'EX_h2o', 'EX_h',
                        'EX_nh4', 'EX_pi', 'EX_so4', 'EX_k']
# gases / nutrients assumed not to be limiting, so their concentration is
# not tracked (in contrast to glucose).

model.objective = 'EX_biomass'
# growth is the objective: during a batch cultivation the cells follow their
# evolutionary objective and grow as fast as possible.

### 2.2 Run dFBA at low HEMEOX activity

We force a flux of at least 0.1 mmol/gDCW/h through heme oxygenase; without
this the model makes no biliverdin (see the production envelope in
Question 2).

In [ ]:
model.reactions.HEMEOX.lower_bound = 0.1

### Question 4

**Question 4: Describe the biliverdin titer and yield. Is the yield the same as the maximum biochemical yield? If not, why not?**

In [ ]:
conc_matrix, exc_names, time_vec, biomass_vec = run_dynamic_fba(
    model, substrate_rxns, init_concentrations, init_biomass,
    time_step, n_steps, excl_uptake_rxns)

glc = conc_matrix[exc_names.index('EX_glc')]
bil = conc_matrix[exc_names.index('EX_biliverdin')]

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
ax[0].plot(time_vec, biomass_vec); ax[0].set_title('Biomass')
ax[0].set_xlabel('Time (h)'); ax[0].set_ylabel('Concentration (g/L)')
ax[1].plot(time_vec, bil); ax[1].set_title('Biliverdin')
ax[1].set_xlabel('Time (h)'); ax[1].set_ylabel('Concentration (mmol/L)')
ax[2].plot(time_vec, glc); ax[2].set_title('Glucose')
ax[2].set_xlabel('Time (h)'); ax[2].set_ylabel('Concentration (mmol/L)')
plt.tight_layout(); plt.show()

print(f'Final biliverdin titer:        {bil[-1]:.4f} mmol/L')
print(f'Residual glucose concentration: {glc[-1]:.4f} mmol/L')
print(f'Glucose consumed:               {init_concentrations - glc[-1]:.4f} mmol/L')
print(f'Yield Yp/s:                     {bil[-1] / (init_concentrations - glc[-1]):.4f} mol/mol')

### 2.3 Run dFBA at higher HEMEOX activities

Now simulate a higher `HEMEOX` expression level. Higher expression does not
necessarily give a higher flux, but for simulation purposes we raise the
lower bound of `HEMEOX` to 0.5 mmol/gDCW/h. **Prepare the code yourself** —
copy the dFBA call above, change the forced `HEMEOX` flux to 0.5, and compare
the titers and yields.

In [ ]:
# Copy the code used for the first dFBA simulation and modify it to force a
# HEMEOX flux of 0.5. Repeat the dFBA and compare the titers and yields with
# the previous simulation.

### Question 5

**Question 5: Compare and comment on the final biliverdin titers and yields reached with a minimum HEMEOX flux of 0.1 mmol/gDCW/h (Question 4) and 0.5 mmol/gDCW/h (this question).**

### 2.4 Scan a range of HEMEOX activities

Let's explore a range of `HEMEOX` activities and see whether the biliverdin
titer always increases. The loop below runs dFBA repeatedly with increasing
lower bounds on `HEMEOX` and records the maximum biliverdin concentration
reached in each cultivation. (This takes a moment — 20 dFBA runs.)

In [ ]:
hemeox_flux = []
biliverdin_max = []
for i in range(1, 21):                 # 20 iterations
    x = i * 0.05                       # 0.05, 0.10, ..., 1.00
    model.reactions.HEMEOX.lower_bound = x
    conc_matrix, exc_names, time_vec, biomass_vec = run_dynamic_fba(
        model, substrate_rxns, init_concentrations, init_biomass,
        time_step, n_steps, excl_uptake_rxns)
    bil = conc_matrix[exc_names.index('EX_biliverdin')]
    hemeox_flux.append(x)
    biliverdin_max.append(bil.max())

plt.figure(figsize=(6, 4))
plt.plot(hemeox_flux, biliverdin_max, 'o-')
plt.title('Biliverdin production')
plt.xlabel('HEMEOX flux (mmol/(gDW h))')
plt.ylabel('Concentration (mmol/L)')
plt.tight_layout(); plt.show()

best = hemeox_flux[int(np.argmax(biliverdin_max))]
print(f'Highest biliverdin titer at HEMEOX = {best:.2f} mmol/gDCW/h')

### Question 6

**Question 6: At what HEMEOX activity do we get the highest biliverdin production? Can you explain what is causing this?**

## 3. Run FSEOF for overexpression targets

If you express `HEMEOX` in *E. coli* and see some production, which genes
should you overexpress to push production further? **FSEOF** (Flux Scanning
based on Enforced Objective Flux) identifies reactions whose flux correlates
with a shift from biomass formation toward product formation (the two compete
for the same carbon). We reload the model first.

In [ ]:
model = read_sbml_model('iECD_1391_biliverdin.xml')

### 3.1 Run FSEOF

We use raven-python's `fseof` (the RAVEN `FSEOF` port). It enforces an
increasing biliverdin flux while maximising biomass (parsimonious LP), and
reports the reactions whose flux is amplified. The **slope** indicates
how strongly a reaction's flux is coupled to increased biliverdin production:
reactions directly involved with biliverdin have a slope of 1; a slope above
1 means the reaction must run multiple times per biliverdin produced. A
higher slope therefore flags reactions carrying most flux for biliverdin
production — candidate overexpression targets. The **direction** is +1
(forward) or -1 (reverse) for reversible reactions, and 0 for irreversible
reactions.

In [ ]:
fseof_result = fseof(model, 'EX_biliverdin', biomass_rxn='EX_biomass',
                     n_steps=10, max_fraction=0.9)
# Over-expression (amplification) targets, ranked by slope:
fseof_result.amplification.to_csv('FSEOF_biliverdin.tab', sep='\t', index=False)
fseof_result.amplification

A number of reactions have a slope close to 8. Look at the pathways
involved in heme biosynthesis here:
[PWY0-1415](https://ecocyc.org/ECOLI/NEW-IMAGE?type=PATHWAY&object=PWY0-1415)
and
[PWY-5188](https://ecocyc.org/ECOLI/NEW-IMAGE?type=PATHWAY&object=PWY-5188).

### Question 7

**Question 7: Can you explain why some of the reactions have a slope of around 8? Why are these slopes not *exactly* 8? Think about how the FSEOF algorithm works.**

## 4. optKnock for knockout targets

To find knockout targets that improve production by enforcing growth
coupling, you can use e.g. optKnock. We will **not** apply it to biliverdin:
in this model no combination of up to three knockouts gives growth-coupled
biliverdin production. Instead we look at **acetate**, **succinate** and
**formate**, for which single knockouts *can* give growth-coupled production.

The algorithm we use is **not** the real optKnock MILP. `runSimpleOptKnock`
runs FBA for each knockout one at a time — a brute-force approach that is slow
for many simultaneous knockouts, but simple and works with the free GLPK
solver. It also works on *reactions* rather than genes (easier to implement).
The text says "knockout of reactions", but in reality genes would be knocked
out.

To speed things up we do not knock out reactions that (a) are essential,
(b) never carry flux, (c) are closely tied to biomass, or (d) are transport
and exchange reactions. This leaves 195 reactions, knocked out one by one.

In [ ]:
model = read_sbml_model('iECD_1391.xml')
selected_rxns = [str(x[0]) for x in sio.loadmat('knockout_SelectedRxns.mat')['selectedRxns'].ravel()]
print(f'Loaded {len(selected_rxns)} reactions selected for knockout')

### 4.1 Run simple optKnock analysis

We define a `run_simple_optknock` helper mirroring RAVEN's function. For each
candidate it knocks out the reaction, maximises biomass, and records the
strategy if growth stays above a threshold *and* the product flux at maximum
growth beats the wild type (i.e. production is growth-coupled). `score` =
growth × production: high production without a drastic growth penalty. Below
we analyse **succinate** (`EX_succ`).

In [ ]:
def run_simple_optknock(model, target_rxn, biomass_rxn, deletions,
                        max_num_ko=1, min_growth=0.05):
    """Brute-force growth-coupling knockout search (mirrors RAVEN
    runSimpleOptKnock). Returns a DataFrame of knockout strategies."""
    model = model.copy()
    model.objective = biomass_rxn
    sol_wt = model.optimize()
    wt_score = sol_wt.fluxes[target_rxn] * sol_wt.objective_value
    found = {}

    def recurse(m, fixed, depth, min_score):
        for rid in deletions:
            if rid in fixed:
                continue
            with m:
                m.reactions.get_by_id(rid).bounds = (0, 0)
                sol = m.optimize()
                if sol.status != 'optimal' or sol.objective_value is None:
                    continue
                growth = sol.objective_value
                prod = sol.fluxes[target_rxn]
                if prod < 1e-10:
                    prod = 0.0
                score = growth * prod
                if growth > min_growth and score > min_score * 1.01:
                    ko = tuple(sorted(fixed + [rid]))
                    if ko not in found or score > found[ko][2]:
                        found[ko] = (growth, prod, score)
                if depth > 1 and growth > min_growth:
                    recurse(m, fixed + [rid], depth - 1, score)

    recurse(model, [], max_num_ko, wt_score)
    cols = ['KO', 'growthRate', 'prodRate', 'score']
    rows = [{'KO': list(ko), 'growthRate': g, 'prodRate': p, 'score': s}
            for ko, (g, p, s) in found.items()]
    df = pd.DataFrame(rows, columns=cols)
    if not df.empty:
        df = df.sort_values('score', ascending=False).reset_index(drop=True)
    return df, sol_wt.objective_value

In [ ]:
targets, wt_growth = run_simple_optknock(
    model, 'EX_succ', 'EX_biomass', selected_rxns, max_num_ko=1)
print(f'Growth rate without knockouts: {wt_growth:.4f}')
targets

For succinate there is only one reaction whose knockout gives
growth-coupled production. `KO` lists the deleted reaction(s); `growthRate`
and `prodRate` are the resulting growth and production rates; `score` is their
product. Note that the maximum growth rate has decreased in the knockout
strain compared with the unmodified model above.

### 4.2 Confirm with production envelope

If the knockout truly gives growth-coupled production, this should be visible
in a production envelope:

In [ ]:
model_succ = model.copy()
model_succ.reactions.get_by_id(targets.KO[0][0]).bounds = (0, 0)   # knock out the target
production_envelope(model_succ, 'EX_succ', 'EX_biomass', n_pts=50)

### Question 8

**Question 8: How does the production envelope show that there is growth-coupled production?**

Having visually confirmed growth-coupled production, you could inspect the
fluxes with and without the knockout to work out *why* the coupling occurs.
(This is not straightforward — feel free to try!)

### 4.3 Further improvement?

Additional knockouts might improve the coupling further (improvement = higher
*score*). Let's first rerun the search on the model where the single target is
already knocked out, to look for a beneficial second knockout:

In [ ]:
targets2, _ = run_simple_optknock(
    model_succ, 'EX_succ', 'EX_biomass', selected_rxns, max_num_ko=1)
print('Number of additional single knockouts found:', len(targets2))
targets2

Once the first target is knocked out, no single second knockout improves the
growth-coupled production. Let's instead look at **three simultaneous
knockouts**. Testing all triples from the full 195-reaction list would take
many hours, so — as a shortcut — we use a small, pre-selected list of 18
reactions. (This is "cheating": the list was chosen because it is known to
contain reactions that are beneficial together.) This search takes a couple of
minutes.

In [ ]:
selected_rxns_small = ['PSP_L', 'FUM', 'PYK', 'PDH', 'POR5', 'ENO', 'HEX1',
                       'PGI', 'PFK', 'FBA', 'TPI', 'PGK', 'PGM', 'LDH_D',
                       'PFL', 'ACKr', 'GNK', 'PGCD']
targets3, _ = run_simple_optknock(
    model, 'EX_succ', 'EX_biomass', selected_rxns_small, max_num_ko=3)
print(f'{len(targets3)} unique knockout sets found')
targets3.head(10)

Let's look at the best-scoring knockout sets:

In [ ]:
top_score = targets3['score'].max()
targets3[targets3['score'] > top_score * 0.999]

There are two sets of three knockouts that reach the same top score (the two
differ only in one reaction; both belong to serine biosynthesis, so they have
the same effect). Compare this top score with the score reached by the single
knockout in Section 4.2.

Now compare the production envelopes. The x-axis is auto-scaled, which makes
direct comparison tricky — keep that in mind, or fix the axes. **Implement one
of the triple knockouts and draw its production envelope** (copy the code from
Section 4.2, but knock out three reactions).

In [ ]:
# Copy the code from Section 4.2 and adapt it to knock out one of the
# best-scoring sets of three reactions, then draw the production envelope.

### Question 9

**Question 9: Describe how the production envelope changed comparing the triple knockout with the earlier single knockout. Is the optimal solution reached after the triple knockout also part of the solution space of the single knockout?**

## Other targets

If you finish early, repeat Section 4 for **formate** (`EX_for`) instead of
succinate. You will get similar results, with different knockouts.

## Notes on differences from the RAVEN version

- **Helper functions.** cobrapy has no direct equivalents of RAVEN's
  `runProductionEnvelope`, `runDynamicFBA`, `FSEOF` or `runSimpleOptKnock`, so
  we reimplemented them above. They follow the same algorithms, so the
  numerical results match the RAVEN answer key (max yield 1.3295, dFBA titers
  ≈0.126 / ≈0.198 mmol/L, FSEOF slopes ≈8, the same growth-coupling
  knockouts).
- **`solveLP(model, 1)` → `pfba`.** RAVEN's `solveLP` with the flux-
  minimisation flag corresponds to cobrapy's parsimonious FBA (`pfba`):
  maximise the objective, then minimise the sum of absolute fluxes. FSEOF
  relies on this (the flux distribution must be unique), so we use `pfba`
  there. In dynamic FBA the growth objective already pins the relevant
  fluxes, so plain FBA gives identical results much faster — that is what we
  use. Plain FBA (`model.optimize()`) also matches RAVEN's `solveLP(model, 0)`
  used by the production envelope and optKnock.
- **Identifiers.** The SBML reader strips the `M_`/`R_` prefixes, so cobrapy
  identifiers equal the RAVEN ones (`pheme_c`, `EX_glc`, `HEMEOX`, ...).
- **`runSimpleOptKnock` duplicates.** The RAVEN function can report duplicate
  knockout sets; our helper de-duplicates them, so the table is shorter but
  contains the same unique strategies.
- **Solver.** cobrapy uses optlang and defaults to GLPK when Gurobi/CPLEX are
  absent. Switch with `model.solver = 'gurobi'`.